In [1]:
import os
import pathlib
import sys
import time

import polars

DNSCBOR_EVAL_DIR = pathlib.Path.cwd()
os.environ["DNSCBOR_EVAL_DIR"] = str(DNSCBOR_EVAL_DIR)

if str((DNSCBOR_EVAL_DIR / "..").absolute()) not in sys.path:
    sys.path.append(str((DNSCBOR_EVAL_DIR / "..").absolute()))

from utils import list_code

# Prepararations for Common IP Prefixes and Name Suffixes Analysis

## Extract Addresses and Names from IoT and Tranco Datasets

For this, you require the `dns_data_*.csv.gz` in `04_cbor4dns_eval/output_datasets/` as generated in [“Extract DNS Data from PCAPs“ in the dataset collection notebook](./01_dataset_collection.ipynb#Extract-DNS-Data-from-PCAPs).

### Extract Addresses

In [2]:
list_code(DNSCBOR_EVAL_DIR / "pair_addrs.py")

#! /usr/bin/env python3
#
# Copyright (C) 2024 TU Dresden
#
# Distributed under terms of the MIT license.

import argparse
import csv
import gzip
import ipaddress
import multiprocessing
import os.path
import sys
import traceback

import bounded_pool_executor
import dns.rdatatype

import cbor4dns_utils


ADDR_COLS = [
    "_ws.col.Source",
    "_ws.col.Destination",
    "dns.a",
    "dns.aaaa",
    "dns.apl.afdpart.ipv4",
    "dns.apl.afdpart.ipv6",
    "dns.ilnp.l32",
    "dns.ipseckey.gateway_ipv4",
    "dns.ipseckey.gateway_ipv6",
    "dns.svcb.svcparam.ipv4hint.ip",
    "dns.svcb.svcparam.ipv6hint.ip",
    "dns.wins.wins_server",
    "dns.wks.address",
    "dns.xpf.destination_ipv4",
    "dns.xpf.destination_ipv6",
    "dns.xpf.source_ipv4",
    "dns.xpf.source_ipv6",
]


WRITER = None


def pair_addrs(row):
    try:
        for i, col1 in enumerate(ADDR_COLS):
            if not row[col1]:
                continue
            for col2 in ADDR_COLS[i:]:
                if not row[col2]:
                    continue
                if isinstance(row[col2], str):
                    # col1 is changed by this col2 change as well
                    row[col2] = [
                        ipaddress.ip_address(addr).packed
                        for addr in row[col2].split("|")
                    ]
                if col1 == col2 or (
                    col1 == "_ws.col.Source" and col2 == "_ws.col.Destination"
                ):
                    continue
                for c1, c2 in [(col1, col1), (col1, col2), (col2, col2)]:
                    if c1.startswith("dns.") and c2.startswith("dns."):
                        prot = row["frame.protocols"].split(":")[-1]
                    else:
                        prot = "xlayer"
                    for j1, addr1 in enumerate(row[c1]):
                        for j2, addr2 in enumerate(row[c2]):
                            if len(addr1) != len(addr2):
                                continue
                            if c1 == c2 and j1 >= j2:
                                # skip duplicate occurrences
                                continue
                            out_row = {
                                "dataset": row["dataset"],
                                "pcap": row["pcap"],
                                "frame": row["frame.number"],
                                "protocol": prot,
                                "msg": "r"
                                if row["dns.flags.response"] in ["1", "True"]
                                else "q",
                                "Field x": c1,
                                "Field y": c2,
                                "Address x": addr1.hex(),
                                "Address y": addr2.hex(),
                                "Common Prefix Bytes": len(
                                    os.path.commonprefix([addr1, addr2])
                                ),
                            }
                            for query_type in row["dns.qry.type"].split("|"):
                                type_name = dns.rdatatype.RdataType(
                                    int(query_type)
                                ).name
                                out_row["qtype"] = type_name
                                WRITER.writerow(out_row)
    except Exception as e:
        print(traceback.format_exc(), file=sys.stderr)
        print("Error:", e, "on", row, file=sys.stderr)


def main():
    global WRITER
    parser = argparse.ArgumentParser()
    parser.add_argument(metavar="<Input CSV filename>", dest="csv")
    args = parser.parse_args()
    if args.csv.endswith(".gz"):
        open_func = gzip.open
    else:
        open_func = open

    # use maximum possible size for field size
    max_int = sys.maxsize
    while True:
        try:
            csv.field_size_limit(max_int)
            break
        except OverflowError:
            max_int = (max_int >> 2)

    with open_func(args.csv, "rt", enc

To start a detached TMUX session running this script **in background of the Docker setup**, run the following command (with the UV setup you might need to step into the virtualenv first, adding, e.g., `. '${DNSCBOR_EVAL_DIR}'/../.env/bin/activate;` before the first `${DNSCBOR_EVAL_DIR}/pair_addrs.py`):

In [3]:
%%bash

tmux new-session -s "pair_addrs" -d \
    "'${DNSCBOR_EVAL_DIR}'/pair_addrs.py \
        '${DNSCBOR_EVAL_DIR}/output_datasets/dns_data_iot.csv.gz' | \
            pigz > '${DNSCBOR_EVAL_DIR}/output_datasets/dns_addrs_iot.csv.gz'; \
     '${DNSCBOR_EVAL_DIR}'/pair_addrs.py \
        '${DNSCBOR_EVAL_DIR}/output_datasets/dns_data_tranco.csv.gz' | \
            pigz > '${DNSCBOR_EVAL_DIR}/output_datasets/dns_addrs_tranco.csv.gz'"

To attach that TMUX session, run the following commant in a Terminal in your Jupyter Lab.

```sh
tmux attach -t "pair_addrs"
```

To kill the TMUX session you can use the following command into a Terminal in your Jupyter Lab.

```sh
tmux send-keys -t "pair_addrs" C-c
```

### Extract Names

In [4]:
list_code(DNSCBOR_EVAL_DIR / "pair_names.py")

#! /usr/bin/env python3
#
# Copyright (C) 2024 TU Dresden
#
# Distributed under terms of the MIT license.

import argparse
import csv
import gzip
import multiprocessing
import os.path
import sys
import traceback

import bounded_pool_executor
import dns.rdatatype

import cbor4dns_utils


NAME_COLS = [
    "dns.qry.name",
    "dns.resp.name",
    "dns.afsdb.hostname",
    "dns.cname",
    "dns.dname",
    "dns.mr",
    "dns.ns",
    "dns.nsec.next_domain_name",
    "dns.ptr.domain_name",
    "dns.rrsig.signers_name",
    "dns.rt.intermediate_host",
    "dns.soa.mname",
    "dns.soa.rname",
    "dns.srv.name",
    "dns.svcb.targetname",
    "dns.winsr.name_result_domain",
]
WRITER = None


def pair_names(row):
    global WRITER
    try:
        for i, col1 in enumerate(NAME_COLS):
            if not row[col1]:
                continue
            for col2 in NAME_COLS[i:]:  # start with same column to apply conversion
                if not row[col2]:
                    continue
                if isinstance(row[col2], str):
                    # col1 is changed by this col2 change as well
                    row[col2] = [name.strip(".") for name in row[col2].split("|")]
                if col1 == col2:
                    continue
                for c1, c2 in [(col1, col1), (col1, col2), (col2, col2)]:
                    for j1, name1 in enumerate(row[c1]):
                        for j2, name2 in enumerate(row[c2]):
                            if c1 == c2 and j1 >= j2:
                                # skip duplicate occurrences
                                continue
                            csb, ccsb = cbor4dns_utils.common_suffixes(name1, name2)
                            out_row = {
                                "dataset": row["dataset"],
                                "pcap": row["pcap"],
                                "frame": row["frame.number"],
                                "protocol": row["frame.protocols"].split(":")[-1],
                                "msg": "r"
                                if row["dns.flags.response"] in ["1", "True"]
                                else "q",
                                "Field x": c1,
                                "Field y": c2,
                                "Name x": name1,
                                "Name y": name2,
                                "Same Name": int(name1 == name2),
                                "Common Suffix Bytes": csb,
                                # add leading delimiter
                                "Common Component Suffix Bytes": ccsb,
                            }
                            for query_type in row["dns.qry.type"].split("|"):
                                type_name = dns.rdatatype.RdataType(
                                    int(query_type)
                                ).name
                                out_row["qtype"] = type_name
                                WRITER.writerow(out_row)
    except Exception as e:
        print(traceback.format_exc(), file=sys.stderr)
        print("Error:", e, "on", row, file=sys.stderr)


def main():
    global WRITER
    parser = argparse.ArgumentParser()
    parser.add_argument(metavar="<Input CSV filename>", dest="csv")
    args = parser.parse_args()
    if args.csv.endswith(".gz"):
        open_func = gzip.open
    else:
        open_func = open

    # use maximum possible size for field size
    max_int = sys.maxsize
    while True:
        try:
            csv.field_size_limit(max_int)
            break
        except OverflowError:
            max_int = (max_int >> 2)

    with open_func(args.csv, "rt", encoding="utf-8") as in_csvfile:
        reader = csv.DictReader(in_csvfile, delimiter=";")
        out_csvfile = sys.stdout
        manager = multiprocessing.Manager()
        WRITER = cbor4dns_utils.FlushableThreadSafeDictWriter(
            out_csvfile,
            manager,
            delimiter=",",
            fieldnames=[
                "dataset"

To start a detached TMUX session running this script **in background of the Docker setup**, run the following command (with the UV setup you might need to step into the virtualenv first, adding, e.g., `. '${DNSCBOR_EVAL_DIR}'/../.env/bin/activate;` before the first `${DNSCBOR_EVAL_DIR}/pair_names.py`):

In [5]:
%%bash

tmux new-session -s "pair_names" -d \
    "'${DNSCBOR_EVAL_DIR}'/pair_names.py \
        '${DNSCBOR_EVAL_DIR}/output_datasets/dns_data_iot.csv.gz' | \
            pigz > '${DNSCBOR_EVAL_DIR}/output_datasets/dns_names_iot.csv.gz'; \
     '${DNSCBOR_EVAL_DIR}'/pair_names.py \
        '${DNSCBOR_EVAL_DIR}/output_datasets/dns_data_tranco.csv.gz' | \
            pigz > '${DNSCBOR_EVAL_DIR}/output_datasets/dns_names_tranco.csv.gz'"

To attach that TMUX session, run the following commant in a Terminal in your Jupyter Lab.

```sh
tmux attach -t "pair_names"
```

To kill the TMUX session you can use the following command into a Terminal in your Jupyter Lab.

```sh
tmux send-keys -t "pair_names" C-c
```

## Extract Name Data from WebPKI and SecSpider Datasets

The WebPKI and SecSpider datasets are not based on PCAP files, so we need dedicated scripts for extracting the name pairs from them. We provide both input files in `04_cbor4dns_eval/input_datasets/` on Zenodo.

### WebPKI Dataset

In [6]:
list_code(DNSCBOR_EVAL_DIR / "collect_from_tls_data.sh")

#!/usr/bin/env bash
#
# Copyright (C) 2024-26 TU Dresden
#
# Distributed under terms of the MIT license.
#

SCRIPT_DIR="$( cd -- "$( dirname -- "${BASH_SOURCE[0]}" )" &> /dev/null && pwd )"

PROCS=$(grep -c '^processor' /proc/cpuinfo)
OUTPUT_DATASETS="${SCRIPT_DIR}/output_datasets"

if [ $# -lt 1 ]; then
    echo "usage: $0 <input ndjson file>" >&2
    exit 1
fi

export INPUT_DATASETS="$(dirname "$(readlink -f "${1}")")"

"${SCRIPT_DIR}"/collect_from_tls_data.py --header
zcat "${1}" | awk 'OFS="\t" {print "'"${1}"'",NR,$0}' | \
    parallel --progress --line-buffer -j"${PROCS}" --spreadstdin --round-robin \
        "${SCRIPT_DIR}"/collect_from_tls_data.py

In [7]:
list_code(DNSCBOR_EVAL_DIR / "collect_from_tls_data.py")

#! /usr/bin/env python3
# vim:fenc=utf-8
#
# Copyright (C) 2024 TU Dresden
#
# Distributed under terms of the MIT license.

import argparse
import collections
import csv
import json
import os
import pathlib
import pprint
import sys
import traceback

from dns.rdatatype import (
    # A,
    # AAAA,
    # CAA,
    CNAME,
    DNAME,
    NS,
    NSEC,
    # NSEC3,
    # RRSIG,
    SOA,
    # TLSA,
    # TXT,
    RdataType,
)

from cbor4dns_utils import decode_name, common_suffixes


SCRIPT_PATH = pathlib.Path(__file__).resolve().parent
INPUT_DATASETS = pathlib.Path(
    os.environ.get("INPUT_DATASETS", SCRIPT_PATH / "input_datasets")
)
CSV_FIELDS = (
    "dataset",
    "input_file",
    "line",
    "protocol",
    "msg",
    "qtype",
    "Field x",
    "Field y",
    "Name x",
    "Name y",
    "Same Name",
    "Common Suffix Bytes",
    "Common Component Suffix Bytes",
)


def collect_rr_name(rr):
    # if rr["type"] in [A, AAAA]:
    #     yield rr["type"].name, ipaddress.ip_address(rr["data"]).packed.hex()
    rr_name = decode_name(rr["name"])
    yield "dns.resp.name", rr_name
    if rr["type"] in [CNAME, DNAME, NS]:
        type_name = RdataType(rr["type"]).name.lower()
        yield f"dns.{type_name}", decode_name(rr["data"])
    elif rr["type"] in [NSEC]:
        yield "dns.nsec.next_domain_name", decode_name(rr["data"].split()[0])
    elif rr["type"] in [SOA]:
        data = rr["data"].split()
        yield "dns.soa.mname", decode_name(data[0])
        yield "dns.soa.rname", decode_name(data[1])
    # TODO CAA records?


def collect_dns_json(dns_json):
    question = dns_json.get("Question", [])
    if len(question) > 1:
        print("==== >1 Question!")
        pprint.pprint(dns_json)
        print("----")
    qtype = None
    for rr in question:
        qtype = RdataType(rr["type"]).name
        yield qtype, "dns.qry.name", decode_name(rr["name"])
    for rr in (
        dns_json.get("Answer", [])
        + dns_json.get("Authority", [])
        + dns_json.get("Additional", [])
    ):
        for rr_type, rr_name in collect_rr_name(rr):
            if rr_type is not None and rr_name is not None:
                yield qtype, rr_type, rr_name


def collect_json(line_file, line_nr, line):
    try:
        d = json.loads(line, object_pairs_hook=collections.OrderedDict)
        res = []
        tranco_name = decode_name(d["domain"])
        for redirect_name, value in d["genesis"]["redirected"]["chains"].items():
            redirect_name = decode_name(redirect_name)
            csb, ccsb = common_suffixes(tranco_name, redirect_name)
            res.append(
                {
                    "dataset": "tls",
                    "input_file": str(line_file),
                    "line": line_nr,
                    "protocol": "http",
                    "Field x": "tranco_base",
                    "Field y": "redirect",
                    "Name x": tranco_name,
                    "Name y": redirect_name,
                    "Same Name": int(tranco_name == redirect_name),
                    "Common Suffix Bytes": csb,
                    "Common Component Suffix Bytes": ccsb,
                }
            )
            for record, dns_json in value["dns"].items():
                if dns_json is None:
                    continue
                dns_names = list(collect_dns_json(dns_json))
                for i1, (qtype1, rtype1, name1) in enumerate(dns_names):
                    csb, ccsb = common_suffixes(tranco_name, name1)
                    res.append(
                        {
                            "dataset": "tls",
                            "input_file": str(line_file),
                            "line": line_nr,
                            "protocol": "assoc",
                            "msg": "r",
                            "qtype": qtype1,
                            "Field x": "tranco_base",
                            "Field y": rtype1,
                            "Name x": tranco_name,
         

To start a detached TMUX session running this script **in background of the Docker setup**, run the following command (with the UV setup you might need to step into the virtualenv first, adding, e.g., `. '${DNSCBOR_EVAL_DIR}'/../.env/bin/activate;` before the first `${DNSCBOR_EVAL_DIR}/collect_from_tls_data.sh`):

In [8]:
%%bash

tmux new-session -s "collect_webpki" -d \
    "'${DNSCBOR_EVAL_DIR}'/collect_from_tls_data.sh \
        '${DNSCBOR_EVAL_DIR}/input_datasets/3.Y5JYG_recertive-lite.out.gz' | \
            pigz > '${DNSCBOR_EVAL_DIR}/output_datasets/dns_names_webpki.csv.gz'"

To attach that TMUX session, run the following commant in a Terminal in your Jupyter Lab.

```sh
tmux attach -t "collect_webpki"
```

To kill the TMUX session you can use the following command into a Terminal in your Jupyter Lab.

```sh
tmux send-keys -t "collect_webpki" C-c
```

### SecSpider Dataset

In [9]:
list_code(DNSCBOR_EVAL_DIR / "collect_from_secspider_data.sh")

#!/usr/bin/env bash
#
# Copyright (C) 2024 TU Dresden
#
# Distributed under terms of the MIT license.
#

SCRIPT_DIR="$( cd -- "$( dirname -- "${BASH_SOURCE[0]}" )" &> /dev/null && pwd )"

PROCS=$(grep -c '^processor' /proc/cpuinfo)
OUTPUT_DATASETS="${SCRIPT_DIR}/output_datasets"

if [ $# -lt 1 ]; then
    echo "usage: $0 <input ndjson file>" >&2
    exit 1
fi

export INPUT_DATASETS="$(dirname "$(readlink -f "${1}")")"

function prep_join() {
    zcat "${1}" | awk '
        BEGIN{FS=OFS="\t"}
        NR == 1 {print "LINE",$0}
        # remove stray "s in dataset while constructing
        NR > 1 {print NR-1,$0}'
}

"${SCRIPT_DIR}"/collect_from_secspider_data.py --header "${1}"
join -t $'\t' --header -j 2 <(prep_join "${1}") <(prep_join "${1}") | \
    parallel --compress --line-buffer -j"${PROCS}" --spreadstdin --round-robin \
        "${SCRIPT_DIR}"/collect_from_secspider_data.py "${1}"

In [10]:
list_code(DNSCBOR_EVAL_DIR / "collect_from_secspider_data.py")

#! /usr/bin/env python3
# vim:fenc=utf-8
#
# Copyright (C) 2024 TU Dresden
#
# Distributed under terms of the MIT license.

import argparse
import concurrent.futures
import csv
import os
import pathlib
import sys
import traceback

from dns.rdatatype import (
    CNAME,
    DNAME,
    NS,
    NSEC,
    SOA,
    RdataType,
)

from cbor4dns_utils import decode_name, common_suffixes


SCRIPT_PATH = pathlib.Path(__file__).resolve().parent
INPUT_DATASETS = pathlib.Path(
    os.environ.get("INPUT_DATASETS", SCRIPT_PATH / "input_datasets")
)
IN_CSV_FIELDS = (
    "name",
    "line1",
    "rr_type1",
    "val1",
    "line2",
    "rr_type2",
    "val2",
)
OUT_CSV_FIELDS = (
    "dataset",
    "input_file",
    "line",
    "protocol",
    "msg",
    "qtype",
    "Field x",
    "Field y",
    "Name x",
    "Name y",
    "Same Name",
    "Common Suffix Bytes",
    "Common Component Suffix Bytes",
)


def convert_rr_data(rr_type, rr_data):
    if rr_type in [CNAME, DNAME, NS]:
        type_name = RdataType(rr_type).name.lower()
        yield f"dns.{type_name}", decode_name(rr_data)
    elif rr_type in [NSEC]:
        yield "dns.nsec.next_domain_name", decode_name(rr_data.split()[0])
    elif rr_type in [SOA]:
        data = rr_data.split()
        yield "dns.soa.mname", decode_name(data[0])
        yield "dns.soa.rname", decode_name(data[1])
    else:
        assert False, f"{rr_type} not supported"


def write_row(writer, out_csvfile, row):
    writer.writerow(row)
    out_csvfile.flush()


def main():
    argparser = argparse.ArgumentParser()
    argparser.add_argument("--header", action="store_true", help="Print header and exit")
    argparser.add_argument("input_filename", type=pathlib.Path)
    args = argparser.parse_args()
    input_filename = args.input_filename.absolute().relative_to(INPUT_DATASETS)
    in_csvfile = sys.stdin
    reader = csv.DictReader(in_csvfile, delimiter="\t", quoting=csv.QUOTE_NONE, fieldnames=IN_CSV_FIELDS)
    out_csvfile = sys.stdout
    writer = csv.DictWriter(out_csvfile, delimiter=",", fieldnames=OUT_CSV_FIELDS)
    if args.header:
        writer.writeheader()
        return
    out_csvfile.flush()
    for row in reader:
        try:
            if row["line1"] == "LINE":
                # skip original header
                continue
            if row["line1"] == row["line2"]:
                # skip same occurrence
                continue
            if not row["rr_type1"] or not row["rr_type2"]:
                print("Error: No pair in", row, file=sys.stderr)
                continue
            for rr_field1, rr_name1 in convert_rr_data(int(row["rr_type1"]), row["val1"]):
                for rr_field2, rr_name2 in convert_rr_data(
                    int(row["rr_type2"]), row["val2"]
                ):
                    resp_name = decode_name(row["name"])
                    csb, ccsb = common_suffixes(resp_name, rr_name1)
                    write_row(
                        writer,
                        out_csvfile,
                        {
                            "dataset": "secspider",
                            "input_file": str(input_filename),
                            "line": row["line1"],
                            "protocol": "dns",
                            "msg": "r",
                            "qtype": "?",
                            "Field x": "dns.resp.name",
                            "Field y": rr_field1,
                            "Name x": resp_name,
                            "Name y": rr_name1,
                            "Same Name": int(resp_name == rr_name1),
                            "Common Suffix Bytes": csb,
                            "Common Component Suffix Bytes": ccsb,
                        },
                    )
                    csb, ccsb = common_suffixes(resp_name, rr_name2)
                    write_row(
                        writer,
                        out_csvfile,
                        {
                            "dataset

To start a detached TMUX session running this script **in background of the Docker setup**, run the following command (with the UV setup you might need to step into the virtualenv first, adding, e.g., `. '${DNSCBOR_EVAL_DIR}'/../.env/bin/activate;` before the first `${DNSCBOR_EVAL_DIR}/collect_from_secspider_data.sh`):

In [11]:
%%bash

tmux new-session -s "collect_secspider" -d \
    "'${DNSCBOR_EVAL_DIR}'/collect_from_secspider_data.sh \
        '${DNSCBOR_EVAL_DIR}/input_datasets/collected_by_pouyan_20231212_martine_secspider.csv.gz' | \
            pigz > '${DNSCBOR_EVAL_DIR}/output_datasets/dns_names_secspider.csv.gz'"

To attach that TMUX session, run the following commant in a Terminal in your Jupyter Lab.

```sh
tmux attach -t "collect_secspider"
```

To kill the TMUX session you can use the following command into a Terminal in your Jupyter Lab.

```sh
tmux send-keys -t "collect_secspider" C-c
```

## Create Histograms
### Address Histograms

In [12]:
list_code(DNSCBOR_EVAL_DIR / "hist_addrs.awk")

BEGIN {
    FS=OFS=",";
    # cpb = common prefix bytes
    print "dataset", "prot", "msg", "qtype", "ip", "bytes", "cpb";
}
$NF ~ /^[0-9]+\s*$/ {
    gsub(/\s+/, "", $(NF));
    if (length($(NF - 1)) == 8) {
        aggr[$1][$4][$5][$6]["ipv4"][$(NF)]++;
    }
    if (length($(NF - 1)) == 32) {
        aggr[$1][$4][$5][$6]["ipv6"][$(NF)]++;
    }
}
END {
    for (dataset in aggr) {
        for (prot in aggr[dataset]) {
            for (msg in aggr[dataset][prot]) {
                for (qtype in aggr[dataset][prot][msg]) {
                    for (ip in aggr[dataset][prot][msg][qtype]) {
                        for (bytes in aggr[dataset][prot][msg][qtype][ip]) {
                            print dataset dataset_marker, prot, msg, qtype, ip, bytes,
                                  aggr[dataset][prot][msg][qtype][ip][bytes];
                        }
                    }
                }
            }
        }
    }
}

To start a detached TMUX session running this script **in background of the Docker setup**, run the following command. It assumes that GNU awk, `gawk`, is installed (as pre-installed in the Docker setup).

In [13]:
%%bash

if tmux ls | grep -q "pair_addrs"; then
    echo "Please wait for TMUX session 'pair_addrs' to finish" >&2
    exit 1
fi

tmux new-session -s "hist_addrs" -d \
    "zcat '${DNSCBOR_EVAL_DIR}/output_datasets/dns_addrs_iot.csv.gz' | \
        gawk -f '${DNSCBOR_EVAL_DIR}/hist_addrs.awk' \
            > '${DNSCBOR_EVAL_DIR}/output_datasets/dns_addrs_hist_iot.csv'; \
     zcat '${DNSCBOR_EVAL_DIR}/output_datasets/dns_addrs_tranco.csv.gz' | \
        gawk -f '${DNSCBOR_EVAL_DIR}/hist_addrs.awk' \
            > '${DNSCBOR_EVAL_DIR}/output_datasets/dns_addrs_hist_tranco.csv'"

To attach that TMUX session, run the following commant in a Terminal in your Jupyter Lab.

```sh
tmux attach -t "hist_addrs"
```

To kill the TMUX session you can use the following command into a Terminal in your Jupyter Lab.

```sh
tmux send-keys -t "hist_addrs" C-c
```

### Name Histograms

In [14]:
list_code(DNSCBOR_EVAL_DIR / "hist_names.awk")

BEGIN {
    FS=OFS=",";
    # csb = common suffix bytes
    # ccsb = common component suffix bytes
    print "dataset", "prot", "msg", "qtype", "bytes", "same_names", "csb", "ccsb";
}
$NF ~ /^[0-9]+\s*$/ && $(NF - 1) ~ /^[0-9]+\s*$/ && $(NF - 2) ~ /^[0-9]+\s*$/ {
    gsub(/\s+/, "", $(NF - 2));
    gsub(/\s+/, "", $(NF - 1));
    gsub(/\s+/, "", $(NF));
    aggr[$1][$4][$5][$6][$(NF - 1)][0] += $(NF - 2);
    aggr[$1][$4][$5][$6][$(NF - 1)][1]++;
    aggr[$1][$4][$5][$6][$(NF)][2]++;
}
END {
    for (dataset in aggr) {
        for (prot in aggr[dataset]) {
            for (msg in aggr[dataset][prot]) {
                for (qtype in aggr[dataset][prot][msg]) {
                    for (bytes in aggr[dataset][prot][msg][qtype]) {
                        print dataset dataset_marker, prot, msg, qtype, bytes,
                              aggr[dataset][prot][msg][qtype][bytes][0],
                              aggr[dataset][prot][msg][qtype][bytes][1],
                              aggr[dataset][prot][msg][qtype][bytes][2];
                    }
                }
            }
        }
    }
}

To start a detached TMUX session running this script **in background of the Docker setup**, run the following command. It assumes that GNU awk, `gawk`, is installed (as pre-installed in the Docker setup).

In [15]:
%%bash

if tmux ls | grep -q -e "pair_names" -e "collect_webpki" -e "collect_secspider"; then
    echo "Please wait for the name pairing TMUX sessions to finish" >&2
    exit 1
fi

tmux new-session -s "hist_names" -d \
    "zcat '${DNSCBOR_EVAL_DIR}/output_datasets/dns_names_iot.csv.gz' | \
        gawk -f '${DNSCBOR_EVAL_DIR}/hist_names.awk' \
           > '${DNSCBOR_EVAL_DIR}/output_datasets/dns_names_hist_iot.csv'; \
     zcat '${DNSCBOR_EVAL_DIR}/output_datasets/dns_names_tranco.csv.gz' | \
        gawk -f '${DNSCBOR_EVAL_DIR}/hist_names.awk' \
           > '${DNSCBOR_EVAL_DIR}/output_datasets/dns_names_hist_tranco.csv'; \
     zcat '${DNSCBOR_EVAL_DIR}/output_datasets/dns_names_webpki.csv.gz' | \
        gawk -f '${DNSCBOR_EVAL_DIR}/hist_names.awk' \
           > '${DNSCBOR_EVAL_DIR}/output_datasets/dns_names_hist_webpki.csv'; \
     zcat '${DNSCBOR_EVAL_DIR}/output_datasets/dns_names_secspider.csv.gz' | \
        gawk -f '${DNSCBOR_EVAL_DIR}/hist_names.awk' \
           > '${DNSCBOR_EVAL_DIR}/output_datasets/dns_names_hist_secspider.csv'"

To attach that TMUX session, run the following commant in a Terminal in your Jupyter Lab.

```sh
tmux attach -t "hist_names"
```

To kill the TMUX session you can use the following command into a Terminal in your Jupyter Lab.

```sh
tmux send-keys -t "hist_names" C-c
```

### Create histogram of suffix occurrences per message

In [16]:
list_code(DNSCBOR_EVAL_DIR / "hist_suffix_occs.awk")

BEGIN {FS=OFS=","}

NR > 1 && $12 > 0 {
    suffix=substr($9,length($9)-$13+1,$13);
    if (suffix ~ /^\./) {
        gsub(/^\.+/, "", suffix)
    }
    # count suffixes per message
    names[$1][$2][$3][$4][$5][$6][suffix]++;
}

END {
    for (d in names) {
        for (f in names[d]) {
            for (n in names[d][f]) {
                for (p in names[d][f][n]) {
                    for (m in names[d][f][n][p]) {
                        for (q in names[d][f][n][p][m]) {
                            for (s in names[d][f][n][p][m][q]) {
                                # create histogram of occurrences per message
                                suffix_occs[d][p][m][q][names[d][f][n][p][m][q][s]]++
                            }
                        }
                    }
                }
            }
        }
    }
    # print histogram
    print "dataset", "protocol", "msg", "qtype", "occurrences", "count";
    for (d in suffix_occs) {
        for (p in suffix_occs[d]) {
            for (m in suffix_occs[d][p]) {
                for (q in suffix_occs[d][p][m]) {
                    for (o in suffix_occs[d][p][m][q]) {
                        print d,p,m,q,o,suffix_occs[d][p][m][q][o];
                    }
                }
            }
        }
    }
}

To start a detached TMUX session running this script **in background of the Docker setup**, run the following command. It assumes that GNU awk, `gawk`, is installed (as pre-installed in the Docker setup).

In [17]:
%%bash

tmux new-session -s "hist_suffix_occs" -d \
    "zcat '${DNSCBOR_EVAL_DIR}'/output_datasets/dns_names_{iot,tranco}.csv.gz | \
        gawk -f '${DNSCBOR_EVAL_DIR}/hist_suffix_occs.awk' | \
           pigz > '${DNSCBOR_EVAL_DIR}/output_datasets/dns_names_hist_suffix_occs.csv.gz'"

To attach that TMUX session, run the following commant in a Terminal in your Jupyter Lab.

```sh
tmux attach -t "hist_suffix_occs"
```

To kill the TMUX session you can use the following command into a Terminal in your Jupyter Lab.

```sh
tmux send-keys -t "hist_suffix_occs" C-c
```

## Prepare Name Match SLD Exploration

In [18]:
list_code(DNSCBOR_EVAL_DIR / "slds_hist.sh")

#!/bin/bash
#
# Copyright (C) 2024 TU Dresden
#
# Distributed under terms of the MIT license.
#

SCRIPT_DIR="$( cd -- "$( dirname -- "${BASH_SOURCE[0]}" )" &> /dev/null && pwd )"

PROCS=${PROCS:-$(grep -c '^processor' /proc/cpuinfo)}
OUTPUT_DATASETS="${OUTPUT_DATASETS:-"${SCRIPT_DIR}/output_datasets"}"

zcat "${OUTPUT_DATASETS}"/dns_names_tranco.csv.gz "${OUTPUT_DATASETS}"/dns_names_iot.csv.gz | \
    "${SCRIPT_DIR}"/slds_hist.py | pigz > "${OUTPUT_DATASETS}"/dns_names_slds_hist_tranco_iot.csv.gz
(
    zcat "${OUTPUT_DATASETS}"/dns_names_slds_hist_tranco_iot.csv.gz | \
        "${SCRIPT_DIR}"/categorize_slds.py --header
    zcat "${OUTPUT_DATASETS}"/dns_names_slds_hist_tranco_iot.csv.gz | \
        parallel -j${PROCS} --pipe --line-buffer "${SCRIPT_DIR}"/categorize_slds.py
) | \
        pigz > "${OUTPUT_DATASETS}"/dns_names_slds_hist_categories_tranco_iot.csv.gz

zcat "${OUTPUT_DATASETS}"/dns_names_secspider.csv.gz "${OUTPUT_DATASETS}"/dns_names_tls.csv.gz | \
    "${SCRIPT_DIR}"/slds_hist.py | pigz > "${OUTPUT_DATASETS}"/dns_names_slds_hist_secspider_tls.csv.gz
(
    zcat "${OUTPUT_DATASETS}"/dns_names_slds_hist_secspider_tls.csv.gz | \
        "${SCRIPT_DIR}"/categorize_slds.py --header
    zcat "${OUTPUT_DATASETS}"/dns_names_slds_hist_secspider_tls.csv.gz | \
        parallel -j${PROCS} --pipe --line-buffer "${SCRIPT_DIR}"/categorize_slds.py
) | \
        pigz > "${OUTPUT_DATASETS}"/dns_names_slds_hist_categories_secspider_tls.csv.gz

In [19]:
list_code(DNSCBOR_EVAL_DIR / "slds_hist.py")

#! /usr/bin/env python3
# vim:fenc=utf-8
#
# Copyright (C) 2024 TU Dresden
#
# Distributed under terms of the MIT license.

import csv
import sys

import publicsuffixlist


HIST = {}
FIELDNAMES = ["dataset", "protocol", "msg", "qtype", "level", "suffix", "count"]


if __name__ == "__main__":
    reader = csv.DictReader(sys.stdin, delimiter=",")
    writer = csv.DictWriter(
        sys.stdout,
        delimiter=",",
        fieldnames=FIELDNAMES
    )
    psl = publicsuffixlist.PublicSuffixList()
    for row in reader:
        try:
            ccsb = int(row["Common Component Suffix Bytes"])
            same_name = bool(int(row["Same Name"]))
        except ValueError:
            # hit a header in the cat'ed files
            continue
        if ccsb == 1:
            suffix = ""
        elif same_name:
            suffix = row["Name y"]
        else:
            suffix = row["Name y"][-(ccsb - 1):]
        sld = psl.privatesuffix(suffix.lower())
        if sld is not None:
            level = "sld"
        else:
            sld = suffix.lower()
            level = "tld" if suffix else ""
        key = (
            row["dataset"],
            row["protocol"],
            row["msg"],
            row["qtype"],
            level,
            sld,
        )
        if key in HIST:
            HIST[key] += 1
        else:
            HIST[key] = 1
    writer.writeheader()
    for key, value in HIST.items():
        writer.writerow(dict(zip(FIELDNAMES, list(key) + [value])))

In [20]:
list_code(DNSCBOR_EVAL_DIR / "categorize_slds.py")

#!/usr/bin/env python3
# vim:fenc=utf-8
#
# Copyright (C) 2024 TU Dresden
#
# Distributed under terms of the MIT license.

import argparse
import csv
import re
import os
import pathlib
import subprocess
import sys


SCRIPT_DIR = pathlib.Path(os.path.dirname(os.path.realpath(__file__)))
DOMAIN_LIST_DATA = SCRIPT_DIR / "v2fly-domain-list-community" / "data"
REGEXES = {}
DOMAINS = {}


def categories_from_file(root, filename, category=None):
    if not category:
        category = filename
    with open(root / filename) as domain_file:
        for line in domain_file:
            line = line.strip()
            line = re.sub(r"\s*#.*$", "", line)
            if not line or line.startswith("keyword:"):
                continue
            elif line.startswith("regexp:"):
                key = re.compile(":".join(line.split(":")[1:]))
                if key in REGEXES:
                    REGEXES[key].add(category)
                else:
                    REGEXES[key] = set([category])
                return
            elif line.startswith("include:"):
                line = line[len("include:"):]
                categories_from_file(root, line, filename)
            elif line.startswith("domain:"):
                line = line[len("domain:"):]
            elif line.startswith("full:"):
                line = line[len("full:"):]
            line = re.sub(r"(\s*@[^\s]+)+$", "", line)
            if line in DOMAINS:
                DOMAINS[line].add(category)
            else:
                DOMAINS[line] = set([category])


def create_category_dicts(prepare=False):
    for root, dirs, files in os.walk(DOMAIN_LIST_DATA):
        for filename in files:
            categories_from_file(pathlib.Path(root), filename)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--header", action="store_true")
    args = parser.parse_args()
    reader = csv.reader(sys.stdin)
    if args.header:
        row = next(reader)
        print(*row[0:6], "categories", *row[6:], sep=";")
        sys.exit(0)
    create_category_dicts()
    writer = csv.writer(sys.stdout, delimiter=";")
    for row in reader:
        if row[6] == "count":
            continue
        row_categories = set()
        for key in REGEXES:
            if key.match(row[5]):
                row_categories.update(REGEXES[key])
        for key in DOMAINS:
            if row[5] == key or row[5].endswith(f".{key}"):
                row_categories.update([cat for cat in DOMAINS[key] if row[5] == key or not re.search(r"\btld\b", cat)])
        writer.writerow(row[0:6] + [list(sorted(row_categories))] + row[6:])

To start a detached TMUX session running this script **in background of the Docker setup**, run the following command (with the UV setup you might need to step into the virtualenv first, adding, e.g., `. '${DNSCBOR_EVAL_DIR}'/../.env/bin/activate;` before the first `${DNSCBOR_EVAL_DIR}/slds_hist.sh`):

In [21]:
%%bash

tmux new-session -s "slds_hist" -d "${DNSCBOR_EVAL_DIR}/slds_hist.sh"

To attach that TMUX session, run the following commant in a Terminal in your Jupyter Lab.

```sh
tmux attach -t "slds_hist"
```

To kill the TMUX session you can use the following command into a Terminal in your Jupyter Lab.

```sh
tmux send-keys -t "slds_hist" C-c
```

### Collapse Categories

Some categories, like subsidary company names or categories primarily used to circumvent censorship are not useful for our analysis. This script collapses these categories to one and also eases parsing for the plotting script.

In [22]:
with open(
    DNSCBOR_EVAL_DIR / "output_datasets" / "dns_names_slds_categories_remove.csv",
    encoding="utf-8",
) as cat_file:
    CATEGORY_REMOVE = set(line.strip() for line in cat_file)

with open(
    DNSCBOR_EVAL_DIR / "output_datasets" / "dns_names_slds_categories_unpair.csv",
    encoding="utf-8",
) as cat_file:
    CATEGORY_UNPAIR = {}
    for line in cat_file:
        line = line.strip().split()
        CATEGORY_UNPAIR[tuple(sorted(set(line[:-1])))] = line[-1]

with open(
    DNSCBOR_EVAL_DIR / "output_datasets" / "dns_names_slds_categories_collapse.csv",
    encoding="utf-8",
) as cat_file:
    COMPANY_COLLAPSE = {}
    for line in cat_file:
        key, value = line.strip().split()
        COMPANY_COLLAPSE[key] = value

start = time.time()
lf = polars.scan_csv(
    DNSCBOR_EVAL_DIR / "output_datasets" / "dns_names_slds_hist_categories_[ts]*.csv.gz",
    separator=";",
    schema_overrides={
        "count": polars.UInt32,
    }
).filter(
    ~polars.col("protocol").is_in(["assoc", "http"])
).with_columns(
    suffix=polars.when(
        polars.col("suffix").is_null()
    ).then(
        polars.lit("No match")
    ).otherwise(
        polars.col("suffix")
    ),
    level=polars.when(
        polars.col("level").is_null()
    ).then(
        polars.lit("root")
    ).otherwise(
        polars.col("level")
    ),
    categories=polars.col("categories").str.replace_all("'", '"').str.json_decode(
        polars.List(polars.String)
    ),
)

for cat_pair in CATEGORY_UNPAIR:
    lf = lf.with_columns(
        categories=polars.when(
            polars.col("categories").list.contains(cat_pair[0])
            & polars.col("categories").list.contains(cat_pair[1])
        ).then(
            polars.col("categories").list.eval(
                polars.element().filter(
                    (polars.element() != CATEGORY_UNPAIR[cat_pair])
                ),
                parallel=True,
            )
        ).otherwise(
            polars.col("categories")
        )
    )
    
lf = lf.with_columns(
    polars.col("categories").list.eval(
        polars.element().filter(
            ~polars.element().is_in(CATEGORY_REMOVE)
            & ~polars.element().str.contains("^(category|geolocation|tld)-")
        ).str.replace("-(ads|cn|dev|pki)$", ""),
        parallel=True,
    )
)

for cat in COMPANY_COLLAPSE:
    lf = lf.with_columns(
        polars.col("categories").list.eval(
            polars.when(
                polars.element() == cat
            ).then(
                polars.lit(COMPANY_COLLAPSE[cat])
            ).otherwise(
                polars.element()
            ),
            parallel=True,
        )
    )

lf = lf.with_columns(
    polars.col("categories").list.unique().list.sort().list.join("/")
).rename({"categories": "category"}).group_by(
    ["dataset", "level", "suffix", "category"]
).agg(occurrences=polars.col("count").sum()).with_columns(
    category=polars.when(
        polars.col("category").str.len_chars() == 0
    ).then(
        None
    ).otherwise(
        polars.col("category")
    )
).with_columns(
    category=polars.when(
        (polars.col("level") == "tld") & polars.col("category").is_null()
    ).then(
        polars.lit("tld")
    ).otherwise(
        polars.when(
            (polars.col("level") == "root")
        ).then(
            polars.lit("No match")
        ).otherwise(
            polars.col("category")
        )
    )
)

lf.sink_csv(
    DNSCBOR_EVAL_DIR / "output_datasets" / "dns_names_slds_hist_categories_datasets.csv",
)
print("Elapsed time:", time.time() - start)

Elapsed time: 711.0734374523163


In [23]:
!pigz -9 "{DNSCBOR_EVAL_DIR}/output_datasets/dns_names_slds_hist_categories_datasets.csv"